# Notebook 06: Vision Transformer (ViT)

**Module:** 10 Transformers  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Understand patch embedding
2. Know ViT architecture end-to-end
3. Implement ViT patch embedding in PyTorch
4. Compare ViT vs CNN inductive bias

## ViT (Dosovitskiy et al., 2020)

**Treat image as sequence of patches:**

1. Split $224 \times 224$ image into $16 \times 16$ patches → $14 \times 14 = 196$ tokens
2. Linear embed each patch → $d_{model}$
3. Prepend [CLS] token
4. Add positional embedding
5. Transformer encoder × L layers
6. [CLS] → MLP head for classification

**Key insight:** With enough data, less inductive bias (no conv) is fine.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, d_model=768):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

pe = PatchEmbed(224, 16, 3, 768)
x = torch.randn(2, 3, 224, 224)
print(f'Patches: {pe(x).shape}')  # (2, 196, 768)

In [ ]:
try:
    import timm
    model = timm.create_model('vit_base_patch16_224', pretrained=False, num_classes=10)
    print(f'timm ViT params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
except ImportError:
    print('Optional: pip install timm for pretrained ViT')

## ViT vs CNN

| | CNN | ViT |
|---|-----|-----|
| Inductive bias | Locality, translation equivariance | None (learned) |
| Data needed | Less | More (or pretrain) |
| Global context | Deep layers | Every layer |
| GeoSpatial | Proven (your UNet++) | Emerging (SegFormer) |

---

## Summary

ViT = patchify image + standard transformer encoder + CLS classification.

**Next:** [07_Swin_Transformer.ipynb](07_Swin_Transformer.ipynb)